# 🧠 Fine-Tuning BanglaBERT on IndoWordNet Dataset for Bengali WSD
### Complete Step-by-Step Cross-Encoder Training & Evaluation Pipeline

This notebook guides you through fine-tuning **BanglaBERT (`csebuetnlp/banglabert`)** as a **Gloss-Matching Cross-Encoder** using our processed **IndoWordNet Dataset** (`data/processed_indowordnet/dataset_splits.json`).

---

### How the Cross-Encoder Works:
1. **Input Pair**: `[CLS] marked_sentence [SEP] target_word : candidate_definition [SEP]`
2. **Contextual Cross-Attention**: The transformer performs full self-attention across both the sentence and the definition simultaneously.
3. **Scoring**: A lightweight linear head scores every candidate sense ($1 \dots K$).
4. **Optimization**: Softmax cross-entropy trains the network to score the true sense highest while suppressing incorrect candidate senses.

---

### Notebook Structure:
* **Step 0**: Environment Setup & Device Configuration (GPU/CPU Auto-detect)
* **Step 1**: Loading the Processed IndoWordNet Dataset
* **Step 2**: Cross-Encoder Pair Construction & Tokenization
* **Step 3**: Initializing the BanglaBERT Model Architecture
* **Step 4**: Untrained Zero-Shot Baseline Evaluation
* **Step 5**: Training Loop with Mixed Precision & Early Stopping
* **Step 6**: Test Set Evaluation & Performance Metrics
* **Step 7**: Live Interactive Inference on New Bengali Sentences


## Step 0: Environment Setup & Hardware Inspection
> **Google Colab Users**: Run this cell to ensure all required libraries (`transformers`, `torch`, `accelerate`) are installed and verify your GPU allocation (e.g. NVIDIA T4).

* **Input**: Python runtime environment.
* **Output**: PyTorch version, active device (`cuda` or `cpu`), GPU VRAM status.


In [ ]:
import os
import sys
import json
import random
import time
from pathlib import Path

# Auto-install required packages if running in Colab / cloud environment
try:
    import transformers
    import torch
except ImportError:
    print("Installing transformers and torch...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "torch", "accelerate"])
    import transformers
    import torch

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda")

print("=== [OUTPUT] Hardware & Environment Status ===")
print(f"PyTorch Version:    {torch.__version__}")
print(f"Transformers:       {transformers.__version__}")
print(f"Active Device:      {device}")
if torch.cuda.is_available():
    print(f"GPU Model:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"Mixed Precision:    {'Enabled (FP16 via torch.cuda.amp)' if use_amp else 'Disabled (CPU mode)'}")


=== [OUTPUT] Hardware & Environment Status ===
PyTorch Version:    2.11.0+cu128
Transformers:       5.16.1
Active Device:      cuda
GPU Model:          Tesla T4
VRAM Available:     14.56 GB
Mixed Precision:    Enabled (FP16 via torch.cuda.amp)


## Step 1: Loading Processed IndoWordNet Dataset
* **Input**: Dataset path `data/processed_indowordnet/dataset_splits.json`.
* **Output**: Vocabulary size, Train/Validation/Test counts, and inspection of a sample training instance.


In [ ]:
# Resolve dataset path (supports running from repo root or notebooks/ folder)
dataset_path = Path("./dataset_splits.json")
if not dataset_path.exists():
    dataset_path = Path("./dataset_splits.json")

with open(dataset_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

catalog = dataset["catalog"]
train_records = dataset["train"]
val_records = dataset["val"]
test_records = dataset["test"]

print(f"=== [OUTPUT] IndoWordNet Dataset Loaded Successfully ===")
print(f"Catalog Words:         {len(catalog):,} polysemous words (0 overlap with initial 100 words)")
print(f"Training Instances:    {len(train_records):,} sentences (70%)")
print(f"Validation Instances:  {len(val_records):,} sentences (15%)")
print(f"Total Sentences:       {len(train_records) + len(val_records) + len(test_records):,}")
print()

# Display a sample training record
sample = train_records[0]
print(f"Sample Training Record:")
print(f"  Target Word:    '{sample['target_word']}'")
print(f"  Ground Truth:   Sense {sample['sense_num']} ({sample['sense_def']})")
print(f"  Input Context:  {sample['text']}")
print(f"  Candidate Senses for '{sample['target_word']}':")
for s_num, s_def in catalog[sample['folder']]['senses'].items():
    prefix = "-> [CORRECT]" if int(s_num) == sample['sense_num'] else "   [OTHER]  "
    print(f"    {prefix} Sense {s_num}: {s_def}")


=== [OUTPUT] IndoWordNet Dataset Loaded Successfully ===
Catalog Words:         3,000 polysemous words (0 overlap with initial 100 words)
Training Instances:    5,990 sentences (70%)
Validation Instances:  1,283 sentences (15%)
Total Sentences:       8,558

Sample Training Record:
  Target Word:    'ক্লান্ত'
  Ground Truth:   Sense 2 (পরিশ্রান্ত, শ্রান্ত (যে ক্লান্ত হয়ে পড়েছে বা যে))
  Input Context:  **ক্লান্ত** পথিক বৃক্ষের ছায়ায় আরাম করছে
  Candidate Senses for 'ক্লান্ত':
       [OTHER]   Sense 1: বিতৃষ্ণ (যার কোনো কাজ, বস্তু, ব্যক্তি প্রভৃতির)
    -> [CORRECT] Sense 2: পরিশ্রান্ত, শ্রান্ত (যে ক্লান্ত হয়ে পড়েছে বা যে)


## Step 2: Cross-Encoder Tokenization & Pair Construction
* **Input**: A single context sentence and candidate sense definitions.
* **Mechanism**:
  - Sentence: ` ভারতে **উত্পন্ন** চা বেশী মাত্রায় বিদেশে রপ্তানি করা হয় `
  - Pair 1: `Sentence [SEP] উৎপন্ন : জাত, উত্পাদিত (যার উত্পত্তি হয়েছে)`
  - Pair 2: `Sentence [SEP] উৎপন্ন : জাত, সঞ্জাত (যে ভূমিষ্ঠ হয়েছে বা জন্মগ্রহণ করেছে)`
* **Output**: Token IDs, tokenized structure, and truncation control.


In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "csebuetnlp/banglabert"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def build_cross_encoder_pairs(context: str, target_word: str, senses: dict):
    first_texts = []
    second_texts = []
    sense_nums = []
    for num, definition in senses.items():
        first_texts.append(context)
        second_texts.append(f"{target_word} : {definition}")
        sense_nums.append(int(num))
    return first_texts, second_texts, sense_nums

# Demonstrate on sample
f_texts, s_texts, s_nums = build_cross_encoder_pairs(
    sample['text'],
    sample['target_word'],
    catalog[sample['folder']]['senses']
)

encoded = tokenizer(
    f_texts, s_texts,
    max_length=256,
    truncation="only_first",
    padding=True,
    return_tensors="pt"
)

print(f"=== [OUTPUT] Cross-Encoder Tokenized Pairs ({len(f_texts)} candidate pairs) ===")
for i, (f, s) in enumerate(zip(f_texts, s_texts), 1):
    print(f"Candidate Pair {i}:")
    print(f"  Sentence Text:   '{f}'")
    print(f"  Sense Definition: '{s}'")
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][i-1])
    print(f"  Input Tokens ({len(tokens)}): {tokens[:14]} ... {tokens[-6:]}\n")


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

=== [OUTPUT] Cross-Encoder Tokenized Pairs (2 candidate pairs) ===
Candidate Pair 1:
  Sentence Text:   '**ক্লান্ত** পথিক বৃক্ষের ছায়ায় আরাম করছে'
  Sense Definition: 'ক্লান্ত : বিতৃষ্ণ (যার কোনো কাজ, বস্তু, ব্যক্তি প্রভৃতির)'
  Input Tokens (28): ['[CLS]', '*', '*', 'ক্লান্ত', '*', '*', 'পথিক', 'বৃক্ষের', 'ছায়ায়', 'আরাম', 'করছে', '[SEP]', 'ক্লান্ত', ':'] ... [',', 'ব্যক্তি', 'প্রভৃতির', ')', '[SEP]', '[PAD]']

Candidate Pair 2:
  Sentence Text:   '**ক্লান্ত** পথিক বৃক্ষের ছায়ায় আরাম করছে'
  Sense Definition: 'ক্লান্ত : পরিশ্রান্ত, শ্রান্ত (যে ক্লান্ত হয়ে পড়েছে বা যে)'
  Input Tokens (28): ['[CLS]', '*', '*', 'ক্লান্ত', '*', '*', 'পথিক', 'বৃক্ষের', 'ছায়ায়', 'আরাম', 'করছে', '[SEP]', 'ক্লান্ত', ':'] ... ['হয়ে', 'পড়েছে', 'বা', 'যে', ')', '[SEP]']



## Step 3: Initializing BanglaBERT Cross-Encoder Architecture
* **Input**: Pretrained `csebuetnlp/banglabert` weights.
* **Architecture**: ELECTRA-base discriminator backbone + Linear projection head (`num_labels = 1`).
* **Output**: Model parameter counts and layer layout.


In [ ]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification

print(f"=== [INPUT] Loading Pretrained Base Model: '{BASE_MODEL}'... ===")

# num_labels=1 outputs a single scalar match score per (context, gloss) pair
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"=== [OUTPUT] Model Architecture Initialized ===")
print(f"Model Type:           {model.config.model_type.upper()} ({BASE_MODEL})")
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Classification Head:  {model.classifier}")


=== [INPUT] Loading Pretrained Base Model: 'csebuetnlp/banglabert'... ===


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


=== [OUTPUT] Model Architecture Initialized ===
Model Type:           ELECTRA (csebuetnlp/banglabert)
Total Parameters:     110,618,113
Trainable Parameters: 110,618,113
Classification Head:  ElectraClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (activation): GELUActivation()
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=1, bias=True)
)


## Step 4: Untrained Zero-Shot Baseline Evaluation
* **Input**: Validation split evaluated on the untrained classification head.
* **Purpose**: Measure initial random baseline accuracy before fine-tuning.
* **Output**: Baseline validation loss and accuracy (~38% random choice across candidate senses).


In [ ]:
def score_batch(model, items, tokenizer, device, max_len=256):
  firsts, seconds, rows, cols = [], [], [], []
  for row, (context, target, senses) in enumerate(items):
    f, s, nums = build_cross_encoder_pairs(context, target, senses)
    firsts += f
    seconds += s
    rows += [row] * len(nums)
    cols += list(range(len(nums)))

  enc = tokenizer(
      firsts,
      seconds,
      max_length=max_len,
      truncation="only_first",
      padding=True,
      return_tensors="pt",
  ).to(device)

  # Explicit .float() ensures FP32 matching matrix dtype
  pair_scores = model(**enc).logits.squeeze(-1).float()

  num_items = len(items)
  max_senses = max(cols) + 1
  matrix = torch.full(
      (num_items, max_senses), float("-inf"), dtype=torch.float32, device=device
  )
  matrix[torch.tensor(rows, device=device), torch.tensor(cols, device=device)] = (
      pair_scores
  )
  return matrix

# Evaluate a small validation sample before training
val_subset = val_records[:100]
val_items = [(r['text'], r['target_word'], catalog[r['folder']]['senses']) for r in val_subset]
val_labels = torch.tensor([r['sense_label'] for r in val_subset], device=device)

model.eval()
logits = score_batch(model, val_items, tokenizer, device)
preds = logits.argmax(dim=-1)
untrained_acc = (preds == val_labels).float().mean().item()

print(f"=== [OUTPUT] Untrained Baseline Performance ===")
print(f"Untrained Accuracy (Random Guessing): {untrained_acc * 100:.2f}%")
print(f"Target Accuracy after Fine-Tuning:    > 85.00%")


=== [OUTPUT] Untrained Baseline Performance ===
Untrained Accuracy (Random Guessing): 39.00%
Target Accuracy after Fine-Tuning:    > 85.00%


## Step 5: Training Loop with Mixed Precision & Early Stopping
* **Hyperparameters**:
  - **Learning Rate**: `2e-5` (AdamW)
  - **Weight Decay**: `0.01`
  - **Batch Size**: `8` (expands to ~20-25 candidate pairs per step)
  - **Warmup Ratio**: `10%` with linear decay
  - **Epochs**: `3`
  - **Mixed Precision**: Automatic FP16 scaling (`torch.cuda.amp`)
* **Input**: 6,073 training sentences and 1,301 validation sentences.
* **Output**: Live step loss, epoch summary table, and best model saved.


In [ ]:
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F

EPOCHS = 3
BATCH_SIZE = 8
LR = 2e-5
WEIGHT_DECAY = 0.01

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = (len(train_records) + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.10), total_steps)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
with torch.amp.autocast('cuda', enabled=use_amp):
  logits = score_batch(model, items, tokenizer, device)
  loss = F.cross_entropy(logits, labels)

checkpoint_dir = Path("../checkpoints/banglabert-indowordnet")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print(f"=== [INPUT] Starting Fine-Tuning on IndoWordNet Dataset ===")
print(f"Training Instances:    {len(train_records):,}")
print(f"Validation Instances:  {len(val_records):,}")
print(f"Steps per Epoch:       {steps_per_epoch}")
print(f"Total Optimizer Steps: {total_steps}\n")

best_val_acc = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    order = list(range(len(train_records)))
    random.shuffle(order)

    for step, start in enumerate(range(0, len(order), BATCH_SIZE), 1):
        batch = [train_records[i] for i in order[start:start + BATCH_SIZE]]
        items = [(r["text"], r["target_word"], catalog[r["folder"]]["senses"]) for r in batch]
        labels = torch.tensor([r["sense_label"] for r in batch], device=device)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = score_batch(model, items, tokenizer, device)
            loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * len(batch)
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += len(batch)

        if step % 250 == 0 or step == steps_per_epoch:
            print(f"  Epoch {epoch} | Step {step}/{steps_per_epoch} | Batch Loss: {loss.item():.4f}")

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation evaluation
    model.eval()
    val_correct, val_total, val_loss_sum = 0, 0, 0.0
    for v_start in range(0, len(val_records), 16):
        v_batch = val_records[v_start:v_start + 16]
        v_items = [(r["text"], r["target_word"], catalog[r["folder"]]["senses"]) for r in v_batch]
        v_labels = torch.tensor([r["sense_label"] for r in v_batch], device=device)
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
            v_logits = score_batch(model, v_items, tokenizer, device)
            v_loss = F.cross_entropy(v_logits, v_labels)
        val_loss_sum += v_loss.item() * len(v_batch)
        val_correct += (v_logits.argmax(dim=-1) == v_labels).sum().item()
        val_total += len(v_batch)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total
    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc})

    print(f"\n>>> Epoch {epoch} Complete ({time.time() - t0:.1f}s) <<<")
    print(f"    Train Loss: {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}%")
    print(f"    Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc * 100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(checkpoint_dir)
        tokenizer.save_pretrained(checkpoint_dir)
        print(f"    -> [SAVED] Best model checkpoint saved to '{checkpoint_dir}' (Val Acc: {val_acc*100:.2f}%)\n")


=== [INPUT] Starting Fine-Tuning on IndoWordNet Dataset ===
Training Instances:    5,990
Validation Instances:  1,283
Steps per Epoch:       749
Total Optimizer Steps: 2247



/tmp/ipykernel_2798/2095080061.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


  Epoch 1 | Step 250/749 | Batch Loss: 0.6946
  Epoch 1 | Step 500/749 | Batch Loss: 0.7297
  Epoch 1 | Step 749/749 | Batch Loss: 0.6947


/tmp/ipykernel_2798/2095080061.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):



>>> Epoch 1 Complete (88.2s) <<<
    Train Loss: 0.9046 | Train Acc: 58.10%
    Val Loss:   0.7413 | Val Acc:   65.08%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    -> [SAVED] Best model checkpoint saved to '../checkpoints/banglabert-indowordnet' (Val Acc: 65.08%)

  Epoch 2 | Step 250/749 | Batch Loss: 0.7314
  Epoch 2 | Step 500/749 | Batch Loss: 0.5957
  Epoch 2 | Step 749/749 | Batch Loss: 0.3322

>>> Epoch 2 Complete (86.5s) <<<
    Train Loss: 0.6516 | Train Acc: 71.49%
    Val Loss:   0.7060 | Val Acc:   67.89%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    -> [SAVED] Best model checkpoint saved to '../checkpoints/banglabert-indowordnet' (Val Acc: 67.89%)

  Epoch 3 | Step 250/749 | Batch Loss: 0.8063
  Epoch 3 | Step 500/749 | Batch Loss: 1.0280
  Epoch 3 | Step 749/749 | Batch Loss: 0.7743

>>> Epoch 3 Complete (83.2s) <<<
    Train Loss: 0.5002 | Train Acc: 78.95%
    Val Loss:   0.7894 | Val Acc:   67.89%


## Step 6: Test Set Evaluation & Performance Metrics
* **Input**: Unseen Test Set (`1,285` sentences).
* **Output**: Overall Test Accuracy, Macro Precision, Recall, and F1-Score.


In [ ]:
# Load the best saved checkpoint
best_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir).to(device)
best_model.eval()

test_correct = 0
test_total = 0
all_preds = []
all_targets = []

for t_start in range(0, len(test_records), 16):
    t_batch = test_records[t_start:t_start + 16]
    t_items = [(r["text"], r["target_word"], catalog[r["folder"]]["senses"]) for r in t_batch]
    t_labels = [r["sense_label"] for r in t_batch]

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
        t_logits = score_batch(best_model, t_items, tokenizer, device)
        batch_preds = t_logits.argmax(dim=-1).cpu().tolist()

    all_preds.extend(batch_preds)
    all_targets.extend(t_labels)
    test_correct += sum(p == t for p, t in zip(batch_preds, t_labels))
    test_total += len(t_batch)

test_acc = test_correct / test_total

print(f"=== [OUTPUT] Final Evaluation on Unseen Test Set ===")
print(f"Total Test Sentences:   {test_total:,}")
print(f"Correct Predictions:    {test_correct:,}")
print(f"Final Test Accuracy:    {test_acc * 100:.2f}%")
print(f"Accuracy Gain over Base: +{(test_acc - untrained_acc) * 100:.2f}%")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_2798/2433084294.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):


=== [OUTPUT] Final Evaluation on Unseen Test Set ===
Total Test Sentences:   1,285
Correct Predictions:    875
Final Test Accuracy:    68.09%
Accuracy Gain over Base: +29.09%


## Step 7: Live Interactive Inference on Bengali Sentences
* **Input**: Arbitrary ambiguous Bengali sentence and candidate senses.
* **Process**: Computes cross-encoder logits and outputs normalized softmax probability percentages for each sense.
* **Output**: Predicted definition, confidence percentage, and inference latency.


In [ ]:
def predict_wsd(sentence: str, target_word: str, candidate_senses: dict):
    t_start = time.time()
    f_texts, s_texts, sense_nums = build_cross_encoder_pairs(sentence, target_word, candidate_senses)
    enc = tokenizer(f_texts, s_texts, padding=True, truncation="only_first", return_tensors="pt").to(device)

    with torch.no_grad():
        logits = best_model(**enc).logits.squeeze(-1)
        probs = torch.softmax(logits, dim=-1).cpu().tolist()

    elapsed_ms = (time.time() - t_start) * 1000
    ranked = sorted(zip(sense_nums, probs), key=lambda x: -x[1])
    top_num, top_prob = ranked[0]

    print(f"=== [INPUT] Target: '{target_word}' ===")
    print(f"Sentence: '{sentence}'\n")
    print(f"=== [OUTPUT] Ranked Predictions (Latency: {elapsed_ms:.1f}ms) ===")
    for rank, (s_num, prob) in enumerate(ranked, 1):
        indicator = ">>> [PREDICTED]" if rank == 1 else "    [OTHER]    "
        print(f"{indicator} Sense {s_num}: {candidate_senses[str(s_num)]} ({prob*100:.1f}%)")
    print("-" * 60)

# Test Sentence 1: Fruit meaning
predict_wsd(
    sentence="সে ফলের দোকান থেকে এক কিলো পাকা আম কিনলো",
    target_word="ফল",
    candidate_senses=catalog[[k for k, v in catalog.items() if v['target_word'] == 'ফল'][0]]['senses']
)

# Test Sentence 2: Result / Consequence meaning
predict_wsd(
    sentence="কঠোর পরিশ্রমের ফল সবসময় ভালোই হয়",
    target_word="ফল",
    candidate_senses=catalog[[k for k, v in catalog.items() if v['target_word'] == 'ফল'][0]]['senses']
)


IndexError: list index out of range

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Helper to resolve senses from catalog or IndoWordNet on the fly
def resolve_candidate_senses(target_word):
    target_clean = target_word.strip()

    # 1. Look up in catalog (3k dataset)
    matched = [v['senses'] for k, v in catalog.items() if v.get('target_word', '').strip() == target_clean]
    if matched:
        return matched[0], "Catalog (IndoWordNet 3k)"

    # 2. Fallback to IndoWordNet directly if word is outside catalog
    if 'pyiwn' in sys.modules:
        try:
            iwn = pyiwn.IndoWordNet(pyiwn.Language.BENGALI)
            synsets = iwn.synsets(target_clean)
            if synsets:
                senses = {}
                for idx, s in enumerate(synsets, 1):
                    other_lemmas = [l.replace('_', ' ').strip() for l in s.lemma_names() if l.strip().lower() != target_clean.lower()]
                    syn_str = ', '.join(other_lemmas[:2]) if other_lemmas else ''
                    gloss_text = s.gloss().strip()
                    if syn_str and gloss_text:
                        senses[str(idx)] = f"{syn_str} ({gloss_text})"
                    elif gloss_text:
                        senses[str(idx)] = gloss_text
                    else:
                        senses[str(idx)] = syn_str or target_clean
                return senses, "IndoWordNet Lexicon (On-the-fly)"
        except Exception:
            pass
    return None, "Not Found"

# Interactive UI Components
sentence_input = widgets.Textarea(
    value="চাষি জমিতে মই দেওয়ার জন্য মই ও দড়ি নিয়ে এলো",
    placeholder="বাংলা বাক্যটি এখানে লিখুন...",
    description="বাক্য:",
    layout=widgets.Layout(width="90%", height="70px")
)

target_input = widgets.Text(
    value="দড়ি",
    placeholder="টার্গেট শব্দটি লিখুন (যেমন: দড়ি, আজ্ঞা, ফল, বল)...",
    description="শব্দ:",
    layout=widgets.Layout(width="50%")
)

check_btn = widgets.Button(
    description="🔍 Check / Disambiguate Sense",
    button_style="success",
    tooltip="Click to predict meaning",
    icon="search",
    layout=widgets.Layout(width="260px", height="38px", margin="8px 0 8px 0")
)

out_box = widgets.Output()

def on_check_clicked(b):
    with out_box:
        clear_output()
        sentence = sentence_input.value.strip()
        target = target_input.value.strip()

        if not sentence or not target:
            print("⚠️ অনুগ্রহ করে বাক্য এবং টার্গেট শব্দ উভয়ই প্রদান করুন।")
            return

        senses, source = resolve_candidate_senses(target)
        if not senses:
            print(f"❌ শব্দ '{target}'-এর জন্য কোনো অর্থ পাওয়া যায়নি। অনুগ্রহ করে অন্য একটি শব্দ টাইপ করুন।")
            return

        t_start = time.time()
        f_texts, s_texts, sense_nums = build_cross_encoder_pairs(sentence, target, senses)
        enc = tokenizer(f_texts, s_texts, padding=True, truncation="only_first", return_tensors="pt").to(device)

        with torch.no_grad():
            logits = best_model(**enc).logits.squeeze(-1)
            probs = torch.softmax(logits, dim=-1).cpu().tolist()

        elapsed_ms = (time.time() - t_start) * 1000
        ranked = sorted(zip(sense_nums, probs), key=lambda x: -x[1])
        top_num, top_prob = ranked[0]

        print("=" * 65)
        print(f"🎯 Target Word: '{target}' | Source: {source}")
        print(f"📝 Context:     '{sentence}'")
        print(f"⏱️ Latency:     {elapsed_ms:.1f} ms")
        print("=" * 65)
        print(f"\n🏆 PREDICTED SENSE: Sense {top_num} ({top_prob * 100:.1f}% Confidence)")
        print(f"   👉 {senses[str(top_num)]}\n")
        print("--- All Candidate Senses Ranked ---")
        for rank, (s_num, prob) in enumerate(ranked, 1):
            bar = "█" * int(prob * 20)
            prefix = ">>> [TOP]" if rank == 1 else "    [   ]"
            print(f"{prefix} Sense {s_num:2d} ({prob * 100:5.1f}%) | {bar:<20} | {senses[str(s_num)]}")
        print("=" * 65 + "\n")

check_btn.on_click(on_check_clicked)

# Display the interactive app
display(widgets.VBox([
    widgets.HTML("<h3>🔎 Interactive Bengali Word Sense Disambiguation</h3>"),
    widgets.HTML("<p>Type any Bengali sentence and target ambiguous word below, then click <b>Check</b>:</p>"),
    target_input,
    sentence_input,
    check_btn,
    out_box
]))

# Run once initially with default example
on_check_clicked(None)